In [2]:
import numpy as np
import pandas as pd

df_clean = pd.read_csv("../data/cleaned_cars.csv")
df_clean["log_price"] = np.log1p(df_clean["price"])
print(df_clean.shape)

(61024, 15)


In [3]:
stats = (df_clean.groupby("make")["price"]
         .agg(["count", "median"]).query("count >= 30")
         .sort_values("median", ascending=False))
stats["median"] = (stats["median"] / 1e6).round(2)
stats

,count,median
make,,
Range Rover,43,45.00
Porsche,44,36.50
Ford,37,31.50
Lexus,120,20.35
Audi,291,13.50
Haval,379,11.40
JAC,133,11.10
BYD,47,9.43
BMW,193,9.00


In [4]:
import sys
sys.path.append("../src")
import features
from sklearn.model_selection import train_test_split

X = df_clean[features.RAW_INPUTS]
y = df_clean["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
y_train_log = np.log1p(y_train)

print("Train:", X_train.shape, "| Test:", X_test.shape)

Train: (48819, 12) | Test: (12205, 12)


In [5]:
prep = features.build_preprocessor()
Xf_train = features.add_features(X_train)
Xf_test = features.add_features(X_test)

Xt = prep.fit_transform(Xf_train)     # learns medians, scaling, categories from TRAIN only
Xv = prep.transform(Xf_test)          # test is only transformed, never fitted

print("Encoded train:", Xt.shape, "| Encoded test:", Xv.shape)
print("Brand tiers:\n", Xf_train["brand_tier"].value_counts())

Encoded train: (48819, 252) | Encoded test: (12205, 252)
Brand tiers:
 brand_tier
Mid          24842
Budget       19531
Upper-mid     2715
Premium       1316
Unlisted       212
Luxury         203
Name: count, dtype: int64
